<a href="https://colab.research.google.com/github/Chaki34/Chaki34/blob/main/python_music_dance_machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 Python Music Dance Machine 🕺

Upload a song and watch a futuristic, glowing visualizer dance to it —
a pulsing central orb, frequency bars, and an exploding particle field,
all reacting to the bass, mids, treble, and beats in your track.

**How to use this notebook:**
1. Run every cell from top to bottom, in order.
2. When you reach the "Upload your music" cell, choose an MP3, WAV, OGG, or M4A file.
3. Wait for the analysis + rendering step (the animation render is the slowest part).
4. Scroll down to see the audio player and the visualizer.

Feel free to tweak the values in the **🎛️ Controls** cell before running the rest.


## 1. Install dependencies

In [1]:
!pip install librosa numpy matplotlib ipywidgets soundfile -q
print("✅ Libraries installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 45.4 MB/s eta 0:00:00
✅ Libraries installed


## 2. Imports

In [2]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")  # headless backend, safe for Colab
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Circle
from IPython.display import display, HTML, Audio

try:
    import librosa
except ImportError as e:
    raise ImportError(
        "❌ librosa failed to import. Please re-run the install cell above, "
        "then Runtime > Restart runtime, then run the cells again."
    ) from e

print("✅ Imports ready")


✅ Imports ready


## 3. 🎛️ Controls

Sensible defaults are set below so the notebook works immediately —
edit any value here before running the rest of the notebook.


In [3]:
NUM_PARTICLES = 150        # number of floating particles in the field
NUM_BARS = 48               # number of frequency bars around the central circle
BEAT_SENSITIVITY = 1.3      # higher = fewer beats detected, lower = more beats flagged
FPS = 20                    # animation frames per second (keep <= 25 for a smooth Colab render)
SPEED = 1.0                 # overall particle speed multiplier
MAX_DURATION_SECONDS = 30   # only the first N seconds are visualized, to keep rendering fast

print(f"🎛️ Particles: {NUM_PARTICLES} | Bars: {NUM_BARS} | Sensitivity: {BEAT_SENSITIVITY} "
      f"| FPS: {FPS} | Speed: {SPEED} | Max duration: {MAX_DURATION_SECONDS}s")


🎛️ Particles: 150 | Bars: 48 | Sensitivity: 1.3 | FPS: 20 | Speed: 1.0 | Max duration: 30s


## 4. Upload your music

In [4]:
def upload_music():
    """Show a Colab upload widget and return the path to the uploaded audio file."""
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("❌ This upload step only works inside Google Colab.")

    print("📁 Choose an audio file (MP3, WAV, OGG, or M4A)...")
    uploaded = files.upload()

    if not uploaded:
        raise ValueError("❌ No file was uploaded. Run this cell again and choose a file.")

    filename = list(uploaded.keys())[0]
    valid_ext = (".mp3", ".wav", ".ogg", ".m4a")
    if not filename.lower().endswith(valid_ext):
        raise ValueError(f"❌ Unsupported file type: '{filename}'. Please upload one of {valid_ext}")

    return filename


AUDIO_PATH = upload_music()
print(f"🎵 Music loaded: {AUDIO_PATH}")


📁 Choose an audio file (MP3, WAV, OGG, or M4A)...


Saving om-namou-shiby-1124.mp3 to om-namou-shiby-1124.mp3
🎵 Music loaded: om-namou-shiby-1124.mp3


## 5. Load the audio

In [5]:
def load_audio(path, max_duration=None):
    """Load an audio file with librosa, handling common failure cases gracefully."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"❌ '{path}' was not found. Please re-run the upload cell.")

    try:
        y, sr = librosa.load(path, sr=22050, mono=True, duration=max_duration)
    except Exception as e:
        raise RuntimeError(
            f"❌ Could not read '{path}'. The file may be corrupted or in an unsupported format.\n"
            f"Details: {e}"
        )

    if len(y) == 0:
        raise RuntimeError(f"❌ '{path}' appears to be empty or silent.")

    full_duration = librosa.get_duration(path=path)
    mins, secs = divmod(int(full_duration), 60)
    print(f"⏱️ Duration: {mins:02d}:{secs:02d}")
    if max_duration and full_duration > max_duration:
        print(f"ℹ️ Visualizing only the first {max_duration}s to keep rendering fast "
              f"(change MAX_DURATION_SECONDS in the controls cell to adjust).")

    return y, sr


y, sr = load_audio(AUDIO_PATH, max_duration=MAX_DURATION_SECONDS)


⏱️ Duration: 00:27


## 6. Analyze the music

Extracts volume, bass/mid/treble energy, a frequency-bar breakdown, and beat pulses — pre-computed once, per frame, so the animation loop itself stays cheap.

In [6]:
def analyze_audio(y, sr, fps, num_bars, beat_sensitivity):
    """Pre-compute one set of normalized visual-control values per animation frame."""
    hop_length = max(1, int(sr / fps))

    # --- overall volume (RMS energy) ---
    rms = librosa.feature.rms(y=y, hop_length=hop_length)[0]

    # --- spectrogram, split into bass / mid / treble bands ---
    S = np.abs(librosa.stft(y, hop_length=hop_length, n_fft=2048))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)

    bass_mask = freqs < 250
    mid_mask = (freqs >= 250) & (freqs < 4000)
    treble_mask = freqs >= 4000

    bass = S[bass_mask].mean(axis=0) if bass_mask.any() else np.zeros(S.shape[1])
    mid = S[mid_mask].mean(axis=0) if mid_mask.any() else np.zeros(S.shape[1])
    treble = S[treble_mask].mean(axis=0) if treble_mask.any() else np.zeros(S.shape[1])

    # --- frequency bars: group the spectrogram rows into num_bars buckets ---
    bin_edges = np.linspace(0, S.shape[0], num_bars + 1).astype(int)
    bars = np.zeros((num_bars, S.shape[1]))
    for i in range(num_bars):
        lo, hi = bin_edges[i], max(bin_edges[i] + 1, bin_edges[i + 1])
        bars[i] = S[lo:hi].mean(axis=0)

    # --- onset strength + beat tracking ---
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length)
    onset_env = np.interp(
        np.linspace(0, len(onset_env) - 1, S.shape[1]), np.arange(len(onset_env)), onset_env
    )

    tempo, beat_frames = librosa.beat.beat_track(
        y=y, sr=sr, hop_length=hop_length, tightness=100 * beat_sensitivity
    )
    beat_flags = np.zeros(S.shape[1])
    for bf in beat_frames:
        if 0 <= bf < len(beat_flags):
            beat_flags[bf] = 1.0

    # widen each beat into a short decaying pulse so it reads visually
    beat_pulse = np.zeros_like(beat_flags)
    pulse_width = 3
    for idx in np.where(beat_flags > 0)[0]:
        for w in range(pulse_width):
            if idx + w < len(beat_pulse):
                beat_pulse[idx + w] = max(beat_pulse[idx + w], 1.0 - w / pulse_width)

    def normalize(arr):
        arr = np.nan_to_num(arr)
        rng = arr.max() - arr.min()
        return (arr - arr.min()) / rng if rng > 1e-8 else np.zeros_like(arr)

    n_frames = min(len(rms), S.shape[1], bars.shape[1], len(onset_env), len(beat_pulse))

    return {
        "n_frames": n_frames,
        "volume": normalize(rms[:n_frames]),
        "bass": normalize(bass[:n_frames]),
        "mid": normalize(mid[:n_frames]),
        "treble": normalize(treble[:n_frames]),
        "bars": np.array([normalize(bars[i][:n_frames]) for i in range(num_bars)]),
        "onset": normalize(onset_env[:n_frames]),
        "beat_pulse": beat_pulse[:n_frames],
        "tempo": float(np.atleast_1d(tempo)[0]),
        "fps": fps,
    }


print("🔎 Analyzing audio (bass / mid / treble / beats)...")
audio_data = analyze_audio(y, sr, FPS, NUM_BARS, BEAT_SENSITIVITY)
n_frames_found = audio_data["n_frames"]
tempo_found = audio_data["tempo"]
print(f"✅ Analysis complete: {n_frames_found} frames | Estimated tempo: {tempo_found:.1f} BPM")


🔎 Analyzing audio (bass / mid / treble / beats)...
✅ Analysis complete: 542 frames | Estimated tempo: 200.1 BPM


## 7. Audio player

In [7]:
def create_audio_player(path):
    print("🎧 Audio player:")
    return Audio(path)


create_audio_player(AUDIO_PATH)


🎧 Audio player:


## 8. Build the visualizer

A glowing central orb (bass + beats), radiating frequency bars (mids), and a particle field (treble speed, beat explosions).

In [10]:
def create_visualizer(audio_data, num_particles, speed, fps):
    """Build a matplotlib FuncAnimation driven by pre-computed audio_data."""

    n_frames = audio_data["n_frames"]
    bars_data = audio_data["bars"]
    num_bars = bars_data.shape[0]

    fig, ax = plt.subplots(figsize=(7, 7), facecolor="black")
    ax.set_facecolor("black")
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-1.6, 1.6)
    ax.set_aspect("equal")
    ax.axis("off")

    # Central glowing circles
    glow_circles = [
        Circle((0, 0), 0.3, color="cyan", alpha=0.05)
        for _ in range(4)
    ]

    for gc in glow_circles:
        ax.add_patch(gc)

    core_circle = Circle(
        (0, 0),
        0.3,
        color="white",
        alpha=0.9
    )

    ax.add_patch(core_circle)

    # Frequency bars
    bar_angles = np.linspace(
        0,
        2 * np.pi,
        num_bars,
        endpoint=False
    )

    bar_lines = []

    for ang in bar_angles:
        line, = ax.plot(
            [],
            [],
            lw=3,
            solid_capstyle="round"
        )
        bar_lines.append((line, ang))

    # Particle system
    rng = np.random.default_rng(42)

    particle_angle = rng.uniform(
        0,
        2 * np.pi,
        num_particles
    )

    particle_radius = rng.uniform(
        0.35,
        1.4,
        num_particles
    )

    particle_speed = rng.uniform(
        0.002,
        0.01,
        num_particles
    )

    particle_size = rng.uniform(
        8,
        30,
        num_particles
    )

    # Initial particle positions
    px = particle_radius * np.cos(particle_angle)
    py = particle_radius * np.sin(particle_angle)

    scatter = ax.scatter(
        px,
        py,
        s=particle_size,
        c=particle_radius,
        cmap="plasma",
        alpha=0.85,
        edgecolors="none"
    )

    title_text = ax.text(
        0,
        1.5,
        "",
        color="white",
        ha="center",
        fontsize=13,
        fontweight="bold"
    )

    cmap = plt.get_cmap("cool")

    def init():
        core_circle.set_radius(0.3)

        # Keep the same number of particle positions and sizes
        scatter.set_offsets(
            np.column_stack([
                particle_radius * np.cos(particle_angle),
                particle_radius * np.sin(particle_angle)
            ])
        )

        scatter.set_sizes(particle_size)

        scatter.set_array(
            np.clip(
                particle_radius / 1.6,
                0,
                1
            )
        )

        title_text.set_text("🎵 DANCING...")

        return (
            [core_circle, scatter, title_text]
            + [l for l, _ in bar_lines]
            + glow_circles
        )

    def update(frame):
        nonlocal particle_radius

        vol = float(audio_data["volume"][frame])
        bass = float(audio_data["bass"][frame])
        treble = float(audio_data["treble"][frame])
        pulse = float(audio_data["beat_pulse"][frame])

        # Central circle reacts to bass and beat
        radius = (
            0.28
            + 0.35 * bass
            + 0.35 * pulse
        )

        core_circle.set_radius(radius)

        core_circle.set_color(
            cmap(
                np.clip(
                    0.3 + 0.6 * vol,
                    0,
                    1
                )
            )
        )

        # Glow effect
        for i, gc in enumerate(glow_circles):

            gc.set_radius(
                radius
                + 0.08 * (i + 1)
                + 0.06 * pulse
            )

            gc.set_alpha(
                np.clip(
                    0.08
                    * (1 - i / len(glow_circles))
                    * (0.4 + vol),
                    0,
                    1
                )
            )

        # Frequency bars
        for i, (line, ang) in enumerate(bar_lines):

            strength = float(
                np.clip(
                    bars_data[i][frame],
                    0,
                    1
                )
            )

            length = (
                0.35
                + 1.0 * strength
            )

            x0 = 0.32 * np.cos(ang)
            y0 = 0.32 * np.sin(ang)

            x1 = (
                0.32 + length
            ) * np.cos(ang)

            y1 = (
                0.32 + length
            ) * np.sin(ang)

            line.set_data(
                [x0, x1],
                [y0, y1]
            )

            line.set_color(
                cmap(
                    0.2 + 0.8 * strength
                )
            )

            line.set_alpha(
                np.clip(
                    0.6 + 0.4 * vol,
                    0,
                    1
                )
            )

        # Particle movement
        explosion_boost = 1 + 6 * pulse

        particle_radius += (
            particle_speed
            * speed
            * (1 + 4 * treble)
            * explosion_boost
        )

        # Respawn particles outside the visualization
        respawn = particle_radius > 1.6

        if respawn.any():

            count = respawn.sum()

            particle_radius[respawn] = rng.uniform(
                0.3,
                0.4,
                count
            )

            particle_angle[respawn] = rng.uniform(
                0,
                2 * np.pi,
                count
            )

        # New particle positions
        px = (
            particle_radius
            * np.cos(particle_angle)
        )

        py = (
            particle_radius
            * np.sin(particle_angle)
        )

        scatter.set_offsets(
            np.column_stack([
                px,
                py
            ])
        )

        scatter.set_array(
            np.clip(
                particle_radius / 1.6,
                0,
                1
            )
        )

        scatter.set_sizes(
            particle_size
            * (0.7 + 0.6 * vol)
        )

        # Status text
        title_text.set_text(
            "🎵 BEAT! 💥"
            if pulse > 0.5
            else "🎵 DANCING..."
        )

        return (
            [core_circle, scatter, title_text]
            + [l for l, _ in bar_lines]
            + glow_circles
        )

    # Create animation
    anim = animation.FuncAnimation(
        fig,
        update,
        frames=n_frames,
        init_func=init,
        interval=1000 / fps,
        blit=True
    )

    plt.close(fig)

    return anim


print("🎨 Building the visualizer animation...")

anim = create_visualizer(
    audio_data,
    NUM_PARTICLES,
    SPEED,
    FPS
)

print("✅ Animation object ready")

🎨 Building the visualizer animation...
✅ Animation object ready


## 9. ✨ Watch it dance

This renders the animation to an embedded HTML5 video — it's the slowest step, especially for more particles/bars or a longer clip.

In [11]:
print("🎬 Rendering animation to video, this can take a minute or two...")
try:
    html_video = anim.to_html5_video()
except Exception as e:
    raise RuntimeError(
        "❌ Rendering failed. Colab usually ships ffmpeg already, but if this keeps failing, "
        "run: !apt-get install -y ffmpeg -q  and try again.\n"
        f"Details: {e}"
    )

display(HTML(html_video))
print("✅ Done! Press play above, and use the audio player in section 7 to listen along.")


🎬 Rendering animation to video, this can take a minute or two...


/tmp/ipykernel_2652/360443664.py:3: UserWarning: Glyph 127925 (\N{MUSICAL NOTE}) missing from font(s) DejaVu Sans.
  html_video = anim.to_html5_video()
/tmp/ipykernel_2652/360443664.py:3: UserWarning: Glyph 128165 (\N{COLLISION SYMBOL}) missing from font(s) DejaVu Sans.
  html_video = anim.to_html5_video()


✅ Done! Press play above, and use the audio player in section 7 to listen along.
